In [1]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")
torch.cuda.set_per_process_memory_fraction(0.5)

Usando dispositivo: cuda


In [2]:
titles_entrenamiento = pd.read_csv('datasets/titles_train.csv', usecols=['id','description', 'imdb_score'])
titles_entrenamiento = titles_entrenamiento[~(titles_entrenamiento['description'].isnull() | titles_entrenamiento['imdb_score'].isnull())]
titles_entrenamiento['id'] = titles_entrenamiento['id'].str.extract(r'(\d+)', expand=False).astype(int)
titles_entrenamiento.set_index('id', inplace=True)
print(titles_entrenamiento.isnull().sum())

titles_entrenamiento

description    0
imdb_score     0
dtype: int64


,description,imdb_score
id,,
127384,"King Arthur, accompanied by his squire, recrui...",8.2
70993,"Brian Cohen is an average young Jewish man, bu...",8.0
190788,12-year-old Regan MacNeil begins to adapt an e...,8.1
14873,When a madman dubbed 'Scorpio' terrorizes San ...,7.7
98978,Two small children and a ship's cook survive a...,5.8
...,...,...
1004011,When a ballroom dancer’s shot at a crucial tou...,2.2
1040816,Three women with totally different lives accid...,5.8
1014599,A beautiful love story that can happen between...,6.9


Asumo que `description` esta completamente en ingles.

### Model

In [3]:
# Using English BERT model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

texto = titles_entrenamiento['description'].iloc[0]
tokens = tokenizer.tokenize(texto)
print(texto)
print(tokens)

King Arthur, accompanied by his squire, recruits his Knights of the Round Table, including Sir Bedevere the Wise, Sir Lancelot the Brave, Sir Robin the Not-Quite-So-Brave-As-Sir-Lancelot and Sir Galahad the Pure. On the way, Arthur battles the Black Knight who, despite having had all his limbs chopped off, insists he can still fight. They reach Camelot, but Arthur decides not  to enter, as "it is a silly place".
['king', 'arthur', ',', 'accompanied', 'by', 'his', 'squire', ',', 'recruits', 'his', 'knights', 'of', 'the', 'round', 'table', ',', 'including', 'sir', 'bed', '##ever', '##e', 'the', 'wise', ',', 'sir', 'lance', '##lot', 'the', 'brave', ',', 'sir', 'robin', 'the', 'not', '-', 'quite', '-', 'so', '-', 'brave', '-', 'as', '-', 'sir', '-', 'lance', '##lot', 'and', 'sir', 'gala', '##had', 'the', 'pure', '.', 'on', 'the', 'way', ',', 'arthur', 'battles', 'the', 'black', 'knight', 'who', ',', 'despite', 'having', 'had', 'all', 'his', 'limbs', 'chopped', 'off', ',', 'insists', 'he', 

In [4]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[2332, 4300, 1010, 5642, 2011, 2010, 21263, 1010, 15024, 2010, 7307, 1997, 1996, 2461, 2795, 1010, 2164, 2909, 2793, 22507, 2063, 1996, 7968, 1010, 2909, 9993, 10994, 1996, 9191, 1010, 2909, 5863, 1996, 2025, 1011, 3243, 1011, 2061, 1011, 9191, 1011, 2004, 1011, 2909, 1011, 9993, 10994, 1998, 2909, 16122, 16102, 1996, 5760, 1012, 2006, 1996, 2126, 1010, 4300, 7465, 1996, 2304, 5000, 2040, 1010, 2750, 2383, 2018, 2035, 2010, 10726, 24881, 2125, 1010, 16818, 2002, 2064, 2145, 2954, 1012, 2027, 3362, 19130, 4140, 1010, 2021, 4300, 7288, 2025, 2000, 4607, 1010, 2004, 1000, 2009, 2003, 1037, 10021, 2173, 1000, 1012]


In [5]:
titles_entrenamiento['description'] = titles_entrenamiento['description'].astype(str).apply(lambda txt: tokenizer.tokenize(txt))
titles_entrenamiento

,description,imdb_score
id,,
127384,"[king, arthur, ,, accompanied, by, his, squire...",8.2
70993,"[brian, cohen, is, an, average, young, jewish,...",8.0
190788,"[12, -, year, -, old, regan, mac, ##neil, begi...",8.1
14873,"[when, a, madman, dubbed, ', sc, ##or, ##pio, ...",7.7
98978,"[two, small, children, and, a, ship, ', s, coo...",5.8
...,...,...
1004011,"[when, a, ballroom, dancer, ’, s, shot, at, a,...",2.2
1040816,"[three, women, with, totally, different, lives...",5.8
1014599,"[a, beautiful, love, story, that, can, happen,...",6.9


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(titles_entrenamiento['description'], titles_entrenamiento['imdb_score'],
                                                  test_size=0.2, random_state=28)

In [7]:
# Load the BERT model
model = BertModel.from_pretrained(model_name)
model.eval()

def get_english_token_embedding(token: str):
    """
    Returns the static embedding for the token using English BERT.
    """
    token_id = tokenizer.convert_tokens_to_ids(token)
    if token_id is None or token_id == tokenizer.unk_token_id:
        print(f"❌ The token '{token}' is not in the English BERT vocabulary.")
        return None
    embedding_vector = model.embeddings.word_embeddings.weight[token_id]
    print(f"✅ Token: '{token}' | ID: {token_id}")
    print(f"Embedding shape: {embedding_vector.shape}")
    return embedding_vector

In [8]:
from torch.utils.data import Dataset

class dataset_token(Dataset):
    def __init__(self, X_tokens, y):
        self.pairs = list(zip(X_tokens, y))
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        token, ys = self.pairs[idx]
        hidden = get_english_token_embedding(token).to(device)
        return {
            'input': hidden,
            'target': ys,
            'input_length': hidden.shape,
            'target_length': 1
    }

train_dataset = dataset_token(X_train, y_train)
val_dataset = dataset_token(X_val, y_val)


In [10]:
from torch.utils.data import DataLoader
BATCH_SIZE = 32

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print('Primer elemento del dataset de entrenamiento:', train_dataset[0])

✅ Token: '['when', 'a', 'teen', 'in', 'rural', 'india', 'discovers', 'a', 'life', '-', 'changing', 'passion', 'for', 'skate', '##boarding', ',', 'she', 'faces', 'a', 'rough', 'road', 'as', 'she', 'follows', 'her', 'dream', 'to', 'compete', '.']' | ID: [2043, 1037, 9458, 1999, 3541, 2634, 9418, 1037, 2166, 1011, 5278, 6896, 2005, 17260, 21172, 1010, 2016, 5344, 1037, 5931, 2346, 2004, 2016, 4076, 2014, 3959, 2000, 5566, 1012]
Embedding shape: torch.Size([29, 768])
Primer elemento del dataset de entrenamiento: {'input': tensor([[-0.0463,  0.0259, -0.0240,  ..., -0.0416,  0.0289,  0.0316],
        [ 0.0152,  0.0082,  0.0043,  ..., -0.0031, -0.0055,  0.0189],
        [-0.0462, -0.0391, -0.0273,  ...,  0.0013, -0.0226, -0.0560],
        ...,
        [ 0.0131,  0.0082, -0.0087,  ...,  0.0159, -0.0078,  0.0182],
        [-0.0444, -0.0065, -0.0531,  ..., -0.0136, -0.0225,  0.0055],
        [-0.0207, -0.0020, -0.0118,  ...,  0.0128,  0.0200,  0.0259]],
       device='cuda:0', grad_fn=<ToCopyBac